In [ ]:
# 1. 环境准备与数据加载
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 通过 rc 参数设置中文字体（Seaborn 推荐做法）
sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 加载 diamonds 数据集：54000 颗钻石的详细信息
df = sns.load_dataset("diamonds")

# 快速查看数据结构
print(df.head())
print(f"\n数据规模：{df.shape}")
print(f"\n数值列统计摘要：")
print(df[["carat", "price", "depth", "table"]].describe())

In [ ]:
# 2. 基础直方图：钻石价格分布
fig, ax = plt.subplots(figsize=(10, 6))

# Seaborn 一行代码画出直方图 + KDE 曲线
sns.histplot(
    data=df,
    x="price",
    bins=50,          # 分成 50 个区间
    kde=True,         # 同时显示 KDE 平滑曲线
    color="skyblue",
    edgecolor="white",
    ax=ax
)

# Matplotlib 精细美化
ax.set_title("钻石价格分布：直方图 + 核密度估计", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("价格 (美元)", fontsize=12)
ax.set_ylabel("数量", fontsize=12)

# 添加均值和中位数参考线
mean_price = df["price"].mean()
median_price = df["price"].median()
ax.axvline(mean_price, color="red", linestyle="--", linewidth=2, label=f"均值: ${mean_price:,.0f}")
ax.axvline(median_price, color="green", linestyle="--", linewidth=2, label=f"中位数: ${median_price:,.0f}")
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("diamond_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 3. 分组对比：不同切工等级的克拉数分布
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 按 cut 分组，分别画出克拉数分布
cuts = ["Ideal", "Premium", "Very Good", "Good"]
colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]

for idx, (cut, color) in enumerate(zip(cuts, colors)):
    row, col = divmod(idx, 2)
    ax = axes[row, col]
    
    subset = df[df["cut"] == cut]
    
    sns.histplot(
        data=subset,
        x="carat",
        bins=30,
        kde=True,
        color=color,
        alpha=0.7,
        ax=ax
    )
    
    ax.set_title(f"{cut} 切工 - 克拉数分布", fontsize=13, fontweight="bold")
    ax.set_xlabel("克拉 (carat)", fontsize=11)
    ax.set_ylabel("数量", fontsize=11)

plt.suptitle("不同切工等级的钻石克拉数分布对比", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("carat_distribution_by_cut.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 4. 叠加 KDE：多组对比一目了然
fig, ax = plt.subplots(figsize=(10, 6))

# 在同一个图上叠加多个 KDE 曲线
for cut, color in zip(cuts, colors):
    subset = df[df["cut"] == cut]
    sns.kdeplot(
        data=subset,
        x="carat",
        label=cut,
        color=color,
        linewidth=2.5,
        fill=True,
        alpha=0.3,
        ax=ax
    )

# Matplotlib 精细美化
ax.set_title("不同切工等级的克拉数密度曲线对比", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("克拉 (carat)", fontsize=12)
ax.set_ylabel("密度", fontsize=12)
ax.legend(title="切工等级", fontsize=11, title_fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("kde_comparison_by_cut.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5. 高级玩法：散点图 + 边缘密度图（Joint Plot 风格）
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 3], hspace=0.05, wspace=0.05)

# 主图：散点图（展示 carat vs price 的关系）
ax_main = fig.add_subplot(gs[1, 0])
sns.scatterplot(
    data=df.sample(3000),  # 抽样 3000 个点避免过度绘制
    x="carat",
    y="price",
    hue="cut",
    palette="Set2",
    alpha=0.6,
    s=30,
    ax=ax_main,
    legend=False
)
ax_main.set_xlabel("")
ax_main.set_ylabel("价格 (美元)", fontsize=12)

# 上方：carat 的 KDE 分布
ax_top = fig.add_subplot(gs[0, 0])
sns.kdeplot(data=df, x="carat", fill=True, color="steelblue", ax=ax_top)
ax_top.set_xlabel("")
ax_top.set_ylabel("")
ax_top.set_xticklabels([])
ax_top.set_yticklabels([])
ax_top.axis("off")

# 右侧：price 的 KDE 分布
ax_right = fig.add_subplot(gs[1, 1])
sns.kdeplot(data=df, y="price", fill=True, color="steelblue", ax=ax_right)
ax_right.set_xlabel("")
ax_right.set_ylabel("")
ax_right.set_xticklabels([])
ax_right.set_yticklabels([])
ax_right.axis("off")

# 隐藏右上角空白
ax_corner = fig.add_subplot(gs[0, 1])
ax_corner.axis("off")

fig.suptitle("克拉数与价格联合分布 + 边缘密度图", fontsize=15, fontweight="bold", y=0.98)
plt.savefig("jointplot_with_marginal_kde.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 6. 完整代码模板（复制即用）
import seaborn as sns
import matplotlib.pyplot as plt

# 基础设置
sns.set_theme(style="whitegrid", font_scale=1.1,
            rc={"font.sans-serif": ["Microsoft YaHei"], 
                "axes.unicode_minus": False})

# 加载数据
df = sns.load_dataset("diamonds")

# 直方图 + KDE 叠加
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x="price", bins=50, kde=True, 
             color="skyblue", edgecolor="white", ax=ax)

# Matplotlib 精细美化
ax.set_title("钻石价格分布", fontsize=16, fontweight="bold")
ax.set_xlabel("价格 (美元)", fontsize=12)
ax.set_ylabel("数量", fontsize=12)

# 添加统计参考线
ax.axvline(df["price"].mean(), color="red", linestyle="--", 
           linewidth=2, label=f"均值: ${df['price'].mean():,.0f}")
ax.axvline(df["price"].median(), color="green", linestyle="--", 
           linewidth=2, label=f"中位数: ${df['price'].median():,.0f}")
ax.legend()

plt.tight_layout()
fig.savefig("diamond_price_distribution.png", dpi=300, bbox_inches="tight")
print("图表已保存为 'diamond_price_distribution.png'")
plt.show()